In [1]:
def affine_alignment(v, w, match, mismatch, gap_open, gap_extend):
    """
    Computes global alignment score and alignment using affine gap penalties.
    v, w: input strings
    match, mismatch: integer scores
    gap_open: opening penalty (positive penalty)
    gap_extend: extension penalty (positive penalty)
    """

    # Score function for characters
    def s(a, b):
        return match if a == b else -mismatch

    n, m = len(v), len(w)

    # Initialize matrices
    M = [[float('-inf')] * (m + 1) for _ in range(n + 1)]
    Ix = [[float('-inf')] * (m + 1) for _ in range(n + 1)]
    Iy = [[float('-inf')] * (m + 1) for _ in range(n + 1)]

    # Backtrack pointers
    backM = [[None] * (m+1) for _ in range(n+1)]
    backIx = [[None] * (m+1) for _ in range(n+1)]
    backIy = [[None] * (m+1) for _ in range(n+1)]

    # Initialization
    M[0][0] = 0

    for i in range(1, n+1):
        Ix[i][0] = -(gap_open + (i-1)*gap_extend)
        backIx[i][0] = ("Ix", i-1, 0)
        M[i][0] = float('-inf')
        Iy[i][0] = float('-inf')

    for j in range(1, m+1):
        Iy[0][j] = -(gap_open + (j-1)*gap_extend)
        backIy[0][j] = ("Iy", 0, j-1)
        M[0][j] = float('-inf')
        Ix[0][j] = float('-inf')

    # Fill matrices
    for i in range(1, n+1):
        for j in range(1, m+1):

            # Match/Mismatch matrix M
            candidates = [
                (M[i-1][j-1], "M"),
                (Ix[i-1][j-1], "Ix"),
                (Iy[i-1][j-1], "Iy")
            ]
            best_score, source = max(candidates, key=lambda x: x[0])
            M[i][j] = best_score + s(v[i-1], w[j-1])
            backM[i][j] = (source, i-1, j-1)

            # Ix: gap in w (horizontal gap)
            open_gap = M[i-1][j] - (gap_open + gap_extend)
            extend_gap = Ix[i-1][j] - gap_extend
            if open_gap >= extend_gap:
                Ix[i][j] = open_gap
                backIx[i][j] = ("M", i-1, j)
            else:
                Ix[i][j] = extend_gap
                backIx[i][j] = ("Ix", i-1, j)

            # Iy: gap in v (vertical gap)
            open_gap = M[i][j-1] - (gap_open + gap_extend)
            extend_gap = Iy[i][j-1] - gap_extend
            if open_gap >= extend_gap:
                Iy[i][j] = open_gap
                backIy[i][j] = ("M", i, j-1)
            else:
                Iy[i][j] = extend_gap
                backIy[i][j] = ("Iy", i, j-1)

    # Find best final score
    final_score, state = max(
        [(M[n][m], "M"), (Ix[n][m], "Ix"), (Iy[n][m], "Iy")],
        key=lambda x: x[0]
    )

    # Backtrack to recover alignment
    aligned_v = []
    aligned_w = []

    i, j = n, m
    current_state = state

    while not (i == 0 and j == 0):
        if current_state == "M":
            source, pi, pj = backM[i][j]
            aligned_v.append(v[i-1])
            aligned_w.append(w[j-1])
        elif current_state == "Ix":
            source, pi, pj = backIx[i][j]
            aligned_v.append(v[i-1])
            aligned_w.append('-')
        else:  # Iy
            source, pi, pj = backIy[i][j]
            aligned_v.append('-')
            aligned_w.append(w[j-1])

        current_state = source
        i, j = pi, pj

    # Reverse alignments
    return final_score, ''.join(reversed(aligned_v)), ''.join(reversed(aligned_w))


# Example usage:
score, a1, a2 = affine_alignment(match=1, mismatch=3, gap_open=2, gap_extend=1,
                                 v="GA", w="GTTA")

print(score)
print(a1)
print(a2)

-2
G--A
GTTA


In [2]:
from math import inf

PAM250 = {'A': {'A': 2, 'C': -2, 'E': 0, 'D': 0, 'G': 1, 'F': -3, 'I': -1, 'H': -1, 'K': -1, 'M': -1, 'L': -2, 'N': 0, 'Q': 0, 'P': 1, 'S': 1, 'R': -2, 'T': 1, 'W': -6, 'V': 0, 'Y': -3}, 
          'C': {'A': -2, 'C': 12, 'E': -5, 'D': -5, 'G': -3, 'F': -4, 'I': -2, 'H': -3, 'K': -5, 'M': -5, 'L': -6, 'N': -4, 'Q': -5, 'P': -3, 'S': 0, 'R': -4, 'T': -2, 'W': -8, 'V': -2, 'Y': 0}, 
          'E': {'A': 0, 'C': -5, 'E': 4, 'D': 3, 'G': 0, 'F': -5, 'I': -2, 'H': 1, 'K': 0, 'M': -2, 'L': -3, 'N': 1, 'Q': 2, 'P': -1, 'S': 0, 'R': -1, 'T': 0, 'W': -7, 'V': -2, 'Y': -4}, 
          'D': {'A': 0, 'C': -5, 'E': 3, 'D': 4, 'G': 1, 'F': -6, 'I': -2, 'H': 1, 'K': 0, 'M': -3, 'L': -4, 'N': 2, 'Q': 2, 'P': -1, 'S': 0, 'R': -1, 'T': 0, 'W': -7, 'V': -2, 'Y': -4}, 
          'G': {'A': 1, 'C': -3, 'E': 0, 'D': 1, 'G': 5, 'F': -5, 'I': -3, 'H': -2, 'K': -2, 'M': -3, 'L': -4, 'N': 0, 'Q': -1, 'P': 0, 'S': 1, 'R': -3, 'T': 0, 'W': -7, 'V': -1, 'Y': -5}, 
          'F': {'A': -3, 'C': -4, 'E': -5, 'D': -6, 'G': -5, 'F': 9, 'I': 1, 'H': -2, 'K': -5, 'M': 0, 'L': 2, 'N': -3, 'Q': -5, 'P': -5, 'S': -3, 'R': -4, 'T': -3, 'W': 0, 'V': -1, 'Y': 7}, 
          'I': {'A': -1, 'C': -2, 'E': -2, 'D': -2, 'G': -3, 'F': 1, 'I': 5, 'H': -2, 'K': -2, 'M': 2, 'L': 2, 'N': -2, 'Q': -2, 'P': -2, 'S': -1, 'R': -2, 'T': 0, 'W': -5, 'V': 4, 'Y': -1}, 
          'H': {'A': -1, 'C': -3, 'E': 1, 'D': 1, 'G': -2, 'F': -2, 'I': -2, 'H': 6, 'K': 0, 'M': -2, 'L': -2, 'N': 2, 'Q': 3, 'P': 0, 'S': -1, 'R': 2, 'T': -1, 'W': -3, 'V': -2, 'Y': 0}, 
          'K': {'A': -1, 'C': -5, 'E': 0, 'D': 0, 'G': -2, 'F': -5, 'I': -2, 'H': 0, 'K': 5, 'M': 0, 'L': -3, 'N': 1, 'Q': 1, 'P': -1, 'S': 0, 'R': 3, 'T': 0, 'W': -3, 'V': -2, 'Y': -4}, 
          'M': {'A': -1, 'C': -5, 'E': -2, 'D': -3, 'G': -3, 'F': 0, 'I': 2, 'H': -2, 'K': 0, 'M': 6, 'L': 4, 'N': -2, 'Q': -1, 'P': -2, 'S': -2, 'R': 0, 'T': -1, 'W': -4, 'V': 2, 'Y': -2}, 
          'L': {'A': -2, 'C': -6, 'E': -3, 'D': -4, 'G': -4, 'F': 2, 'I': 2, 'H': -2, 'K': -3, 'M': 4, 'L': 6, 'N': -3, 'Q': -2, 'P': -3, 'S': -3, 'R': -3, 'T': -2, 'W': -2, 'V': 2, 'Y': -1}, 
          'N': {'A': 0, 'C': -4, 'E': 1, 'D': 2, 'G': 0, 'F': -3, 'I': -2, 'H': 2, 'K': 1, 'M': -2, 'L': -3, 'N': 2, 'Q': 1, 'P': 0, 'S': 1, 'R': 0, 'T': 0, 'W': -4, 'V': -2, 'Y': -2}, 
          'Q': {'A': 0, 'C': -5, 'E': 2, 'D': 2, 'G': -1, 'F': -5, 'I': -2, 'H': 3, 'K': 1, 'M': -1, 'L': -2, 'N': 1, 'Q': 4, 'P': 0, 'S': -1, 'R': 1, 'T': -1, 'W': -5, 'V': -2, 'Y': -4}, 
          'P': {'A': 1, 'C': -3, 'E': -1, 'D': -1, 'G': 0, 'F': -5, 'I': -2, 'H': 0, 'K': -1, 'M': -2, 'L': -3, 'N': 0, 'Q': 0, 'P': 6, 'S': 1, 'R': 0, 'T': 0, 'W': -6, 'V': -1, 'Y': -5}, 
          'S': {'A': 1, 'C': 0, 'E': 0, 'D': 0, 'G': 1, 'F': -3, 'I': -1, 'H': -1, 'K': 0, 'M': -2, 'L': -3, 'N': 1, 'Q': -1, 'P': 1, 'S': 2, 'R': 0, 'T': 1, 'W': -2, 'V': -1, 'Y': -3}, 
          'R': {'A': -2, 'C': -4, 'E': -1, 'D': -1, 'G': -3, 'F': -4, 'I': -2, 'H': 2, 'K': 3, 'M': 0, 'L': -3, 'N': 0, 'Q': 1, 'P': 0, 'S': 0, 'R': 6, 'T': -1, 'W': 2, 'V': -2, 'Y': -4}, 
          'T': {'A': 1, 'C': -2, 'E': 0, 'D': 0, 'G': 0, 'F': -3, 'I': 0, 'H': -1, 'K': 0, 'M': -1, 'L': -2, 'N': 0, 'Q': -1, 'P': 0, 'S': 1, 'R': -1, 'T': 3, 'W': -5, 'V': 0, 'Y': -3}, 
          'W': {'A': -6, 'C': -8, 'E': -7, 'D': -7, 'G': -7, 'F': 0, 'I': -5, 'H': -3, 'K': -3, 'M': -4, 'L': -2, 'N': -4, 'Q': -5, 'P': -6, 'S': -2, 'R': 2, 'T': -5, 'W': 17, 'V': -6, 'Y': 0}, 
          'V': {'A': 0, 'C': -2, 'E': -2, 'D': -2, 'G': -1, 'F': -1, 'I': 4, 'H': -2, 'K': -2, 'M': 2, 'L': 2, 'N': -2, 'Q': -2, 'P': -1, 'S': -1, 'R': -2, 'T': 0, 'W': -6, 'V': 4, 'Y': -2},
           'Y': {'A': -3, 'C': 0, 'E': -4, 'D': -4, 'G': -5, 'F': 7, 'I': -1, 'H': 0, 'K': -4, 'M': -2, 'L': -1, 'N': -2, 'Q': -4, 'P': -5, 'S': -3, 'R': -4, 'T': -3, 'W': 0, 'V': -2, 'Y': 10}}

BLOSUM62 = {
 'A': {'A': 4, 'C': 0, 'D': -2, 'E': -1, 'F': -2, 'G': 0, 'H': -2, 'I': -1, 'K': -1, 'L': -1, 'M': -1, 'N': -2, 'P': -1, 'Q': -1, 'R': -1, 'S': 1, 'T': 0, 'V': 0, 'W': -3, 'Y': -2},
 'C': {'A': 0, 'C': 9, 'D': -3, 'E': -4, 'F': -2, 'G': -3, 'H': -3, 'I': -1, 'K': -3, 'L': -1, 'M': -1, 'N': -3, 'P': -3, 'Q': -3, 'R': -3, 'S': -1, 'T': -1, 'V': -1, 'W': -2, 'Y': -2},
 'D': {'A': -2, 'C': -3, 'D': 6, 'E': 2, 'F': -3, 'G': -1, 'H': -1, 'I': -3, 'K': -1, 'L': -4, 'M': -3, 'N': 1, 'P': -1, 'Q': 0, 'R': -2, 'S': 0, 'T': -1, 'V': -3, 'W': -4, 'Y': -3},
 'E': {'A': -1, 'C': -4, 'D': 2, 'E': 5, 'F': -3, 'G': -2, 'H': 0, 'I': -3, 'K': 1, 'L': -3, 'M': -2, 'N': 0, 'P': -1, 'Q': 2, 'R': 0, 'S': 0, 'T': -1, 'V': -2, 'W': -3, 'Y': -2},
 'F': {'A': -2, 'C': -2, 'D': -3, 'E': -3, 'F': 6, 'G': -3, 'H': -1, 'I': 0, 'K': -3, 'L': 0, 'M': 0, 'N': -3, 'P': -4, 'Q': -3, 'R': -3, 'S': -2, 'T': -2, 'V': -1, 'W': 1, 'Y': 3},
 'G': {'A': 0, 'C': -3, 'D': -1, 'E': -2, 'F': -3, 'G': 6, 'H': -2, 'I': -4, 'K': -2, 'L': -4, 'M': -3, 'N': 0, 'P': -2, 'Q': -2, 'R': -2, 'S': 0, 'T': -2, 'V': -3, 'W': -2, 'Y': -3},
 'H': {'A': -2, 'C': -3, 'D': -1, 'E': 0, 'F': -1, 'G': -2, 'H': 8, 'I': -3, 'K': -1, 'L': -3, 'M': -2, 'N': 1, 'P': -2, 'Q': 0, 'R': 0, 'S': -1, 'T': -2, 'V': -3, 'W': -2, 'Y': 2},
 'I': {'A': -1, 'C': -1, 'D': -3, 'E': -3, 'F': 0, 'G': -4, 'H': -3, 'I': 4, 'K': -3, 'L': 2, 'M': 1, 'N': -3, 'P': -3, 'Q': -3, 'R': -3, 'S': -2, 'T': -1, 'V': 3, 'W': -3, 'Y': -1},
 'K': {'A': -1, 'C': -3, 'D': -1, 'E': 1, 'F': -3, 'G': -2, 'H': -1, 'I': -3, 'K': 5, 'L': -2, 'M': -1, 'N': 0, 'P': -1, 'Q': 1, 'R': 2, 'S': 0, 'T': -1, 'V': -2, 'W': -3, 'Y': -2},
 'L': {'A': -1, 'C': -1, 'D': -4, 'E': -3, 'F': 0, 'G': -4, 'H': -3, 'I': 2, 'K': -2, 'L': 4, 'M': 2, 'N': -3, 'P': -3, 'Q': -2, 'R': -2, 'S': -2, 'T': -1, 'V': 1, 'W': -2, 'Y': -1},
 'M': {'A': -1, 'C': -1, 'D': -3, 'E': -2, 'F': 0, 'G': -3, 'H': -2, 'I': 1, 'K': -1, 'L': 2, 'M': 5, 'N': -2, 'P': -2, 'Q': 0, 'R': -1, 'S': -1, 'T': -1, 'V': 1, 'W': -1, 'Y': -1},
 'N': {'A': -2, 'C': -3, 'D': 1, 'E': 0, 'F': -3, 'G': 0, 'H': 1, 'I': -3, 'K': 0, 'L': -3, 'M': -2, 'N': 6, 'P': -2, 'Q': 0, 'R': 0, 'S': 1, 'T': 0, 'V': -3, 'W': -4, 'Y': -2},
 'P': {'A': -1, 'C': -3, 'D': -1, 'E': -1, 'F': -4, 'G': -2, 'H': -2, 'I': -3, 'K': -1, 'L': -3, 'M': -2, 'N': -2, 'P': 7, 'Q': -1, 'R': -2, 'S': -1, 'T': -1, 'V': -2, 'W': -4, 'Y': -3},
 'Q': {'A': -1, 'C': -3, 'D': 0, 'E': 2, 'F': -3, 'G': -2, 'H': 0, 'I': -3, 'K': 1, 'L': -2, 'M': 0, 'N': 0, 'P': -1, 'Q': 5, 'R': 1, 'S': 0, 'T': -1, 'V': -2, 'W': -2, 'Y': -1},
 'R': {'A': -1, 'C': -3, 'D': -2, 'E': 0, 'F': -3, 'G': -2, 'H': 0, 'I': -3, 'K': 2, 'L': -2, 'M': -1, 'N': 0, 'P': -2, 'Q': 1, 'R': 5, 'S': -1, 'T': -1, 'V': -3, 'W': -3, 'Y': -2},
 'S': {'A': 1, 'C': -1, 'D': 0, 'E': 0, 'F': -2, 'G': 0, 'H': -1, 'I': -2, 'K': 0, 'L': -2, 'M': -1, 'N': 1, 'P': -1, 'Q': 0, 'R': -1, 'S': 4, 'T': 1, 'V': -2, 'W': -3, 'Y': -2},
 'T': {'A': 0, 'C': -1, 'D': -1, 'E': -1, 'F': -2, 'G': -2, 'H': -2, 'I': -1, 'K': -1, 'L': -1, 'M': -1, 'N': 0, 'P': -1, 'Q': -1, 'R': -1, 'S': 1, 'T': 5, 'V': 0, 'W': -2, 'Y': -2},
 'V': {'A': 0, 'C': -1, 'D': -3, 'E': -2, 'F': -1, 'G': -3, 'H': -3, 'I': 3, 'K': -2, 'L': 1, 'M': 1, 'N': -3, 'P': -2, 'Q': -2, 'R': -3, 'S': -2, 'T': 0, 'V': 4, 'W': -3, 'Y': -1},
 'W': {'A': -3, 'C': -2, 'D': -4, 'E': -3, 'F': 1, 'G': -2, 'H': -2, 'I': -3, 'K': -3, 'L': -2, 'M': -1, 'N': -4, 'P': -4, 'Q': -2, 'R': -3, 'S': -3, 'T': -2, 'V': -3, 'W': 11, 'Y': 2},
 'Y': {'A': -2, 'C': -2, 'D': -3, 'E': -2, 'F': 3, 'G': -3, 'H': 2, 'I': -1, 'K': -2, 'L': -1, 'M': -1, 'N': -2, 'P': -3, 'Q': -1, 'R': -2, 'S': -2, 'T': -2, 'V': -1, 'W': 2, 'Y': 7},
}

def local_affine_allignment_peptides(v,w,scoring_matrix,gap_open,gap_ext):

    n,m = len(v), len(w); NEG = -inf

    M = [[0] * (m + 1) for _ in range(n + 1)]
    Ix = [[NEG] * (m + 1) for _ in range(n + 1)]
    Iy = [[NEG] * (m + 1) for _ in range(n + 1)]

    # backtrack pointers, store the source state and coordinates
    bm = [[None] * (m + 1) for _ in range(n + 1)]
    bx = [[None] * (m + 1) for _ in range(n + 1)]
    by = [[None] * (m + 1) for _ in range(n + 1)]

    best_score, best_state, best_pos = 0, 'M', (0, 0)

    # first colon/first row: M = 0, Ix/Iy = -inf (already set before)

    for i in range(1, n+1):
        a = v[i-1]
        for j in range(1, m+1):
            b = w[j-1]
            s = scoring_matrix[a][b]

            # M: local reset + 3 diagnoles
            candM = [
                (0,None),
                (M[i-1][j-1] + s, ('M', i-1, j-1)),
                (Ix[i-1][j-1] + s, ('X', i-1, j-1)),
                (Iy[i-1][j-1] + s, ('Y', i-1, j-1))
            ]
            M[i][j], bm[i][j] = max(candM, key = lambda t:t[0])

            # Ix oppening or extending a horizontal gap in w, while taking a symbol from v
            candX = [
                (M[i-1][j] - gap_open, ('M', i, j-1)),
                (Ix[i-1][j] - gap_ext, ('X', i, j-1))
            ]
            Ix[i][j], bx[i][j] = max(candX, key= lambda t:t[0])

            # Iy oppening or extending a horizontal gap in v, while taking a symbol from 2
            candY = [
                (M[i-1][j] - gap_open, ('M', i-1, j)),
                (Iy[i-1][j] - gap_ext, ('Y', i-1, j))
            ]
            Iy[i][j], by[i][j] = max(candY, key = lambda t: t[0])

            # Store the best scoring cell from the three matrices
            local_best = M[i][j]; state = 'M'

            if Ix[i][j] > local_best:
                local_best, state = Ix[i][j], 'X'
            if Iy[i][j] > local_best:
                local_best, state = Iy[i][j], 'Y'
            if local_best > best_score:
                best_score, best_state, best_pos = local_best, state, (i,j)


    # Backtrack from the global maximum to the first cell in M where there is a 0 (reset)
    # Free Taxi ride
    i,j = best_pos
    v_aln = []; w_aln = []; state = best_state

    while i > 0 and j > 0:
        
        if state == 'M':
            # stop at the reset point
            if M[i][j] == 0 and (bm[i][j] is None or bm[i][j][0] is None):
                break

            src = bm[i][j] 
            st, pi, pj = src
            v_aln.append(v[i-1]); w_aln.append(w[j-1])
            
            i,j,state = pi, pj , st
        
        elif state == 'X':
            
            st, pi, pj = bx[i][j]
            v_aln.append(v[i-1]); w_aln.append('-')
            i,j,state = pi,pj,st

        else: # state == 'Y'
            st, pi, pj = by[i][j]
            v_aln.append('-'); w_aln.append(w[j-1])
            i,j,state = pi,pj,st


        # extra break just in case the next step leads us to a reset(0) point in M
        if state == 'M' and M[i][j] == 0 and (bm[i][j] is None or bm[i][j][0] is None):
            break

    return best_score, ''.join(v_aln[::-1]), ''.join(w_aln[::-1])


if __name__ == '__main__':
    # You can test here, just plug in the variables

    s1 = 'YAFDLGYTCMFPVLLGGGELHIVQKETYTAPDEIAHYIKEHGITYIKLTPSLFHTIVNTASFAFDANFESLRLIVLGGEKIIPIDVIAFRKMYGHTEFINHYGPTEATIGA'
    s2 = 'AFDVSAGDFARALLTGGQLIVCPNEVKMDPASLYAIIKKYDITIFEATPALVIPLMEYIYEQKLDISQLQILIVGSDSCSMEDFKTLVSRFGSTIRIVNSYGVTEACIDS'
    scoring_matrix = PAM250
    gap_open = 2
    gap_ext = 1

    score,a,b = local_affine_allignment_peptides(s1,s2,scoring_matrix,gap_open,gap_ext)
    print(score); print(a); print(b)

141
IIIIIPPPPPIIIIDDDVIIIAAFFFRKKKMYYYYGGGGGGGGGGGGGGGGGHHHHHTTEFINNNNHYYYYYYYGPPPPPPPTEEEEEEATI
-----F----L---Q--VC--E-K--PA--YA---K----------------M----E-KLDI---QI------SC------TL-----STI


In [3]:
def smith_waterman_affine(v, w, match, mismatch, gap_open, gap_extend):
    """
    Computes optimal local alignment (Smith-Waterman) using affine gap penalties.
    v, w: sequences
    match: match reward
    mismatch: mismatch penalty
    gap_open: gap opening penalty σ
    gap_extend: gap extension penalty ε
    """

    def score(a, b):
        return match if a == b else -mismatch

    n, m = len(v), len(w)

    # DP matrices
    M  = [[0]*(m+1) for _ in range(n+1)]
    Ix = [[0]*(m+1) for _ in range(n+1)]
    Iy = [[0]*(m+1) for _ in range(n+1)]

    # Backtracking pointers
    backM  = [[None]*(m+1) for _ in range(n+1)]
    backIx = [[None]*(m+1) for _ in range(n+1)]
    backIy = [[None]*(m+1) for _ in range(n+1)]

    best_score = 0
    best_pos = (0, 0)
    best_matrix = "M"

    # Fill matrices
    for i in range(1, n+1):
        for j in range(1, m+1):

            # M matrix (match/mismatch)
            candidates = [
                (M[i-1][j-1], "M"),
                (Ix[i-1][j-1], "Ix"),
                (Iy[i-1][j-1], "Iy")
            ]
            prev_val, source = max(candidates, key=lambda x: x[0])
            val = prev_val + score(v[i-1], w[j-1])
            if val < 0:
                val = 0
                source = None
            M[i][j] = val
            backM[i][j] = (source, i-1, j-1)

            # Ix matrix (gap in w)
            open_gap = M[i-1][j] - (gap_open + gap_extend)
            extend_gap = Ix[i-1][j] - gap_extend
            if open_gap >= extend_gap:
                prev = ("M", i-1, j)
                val = open_gap
            else:
                prev = ("Ix", i-1, j)
                val = extend_gap

            if val < 0:
                val = 0
                prev = None

            Ix[i][j] = val
            backIx[i][j] = prev

            # Iy matrix (gap in v)
            open_gap = M[i][j-1] - (gap_open + gap_extend)
            extend_gap = Iy[i][j-1] - gap_extend
            if open_gap >= extend_gap:
                prev = ("M", i, j-1)
                val = open_gap
            else:
                prev = ("Iy", i, j-1)
                val = extend_gap

            if val < 0:
                val = 0
                prev = None

            Iy[i][j] = val
            backIy[i][j] = prev

            # Keep track of best cell
            candidates = [(M[i][j], "M"), (Ix[i][j], "Ix"), (Iy[i][j], "Iy")]
            cell_best, cell_matrix = max(candidates, key=lambda x: x[0])

            if cell_best > best_score:
                best_score = cell_best
                best_pos = (i, j)
                best_matrix = cell_matrix

    # Backtracking
    i, j = best_pos
    matrix = best_matrix
    aligned_v = []
    aligned_w = []

    while True:
        if matrix == "M":
            source, pi, pj = backM[i][j]
            if M[i][j] == 0: break
            aligned_v.append(v[i-1])
            aligned_w.append(w[j-1])

        elif matrix == "Ix":
            source, pi, pj = backIx[i][j]
            if Ix[i][j] == 0: break
            aligned_v.append(v[i-1])
            aligned_w.append('-')

        elif matrix == "Iy":
            source, pi, pj = backIy[i][j]
            if Iy[i][j] == 0: break
            aligned_v.append('-')
            aligned_w.append(w[j-1])

        matrix = source
        i, j = pi, pj
        if matrix is None:
            break

    # Reverse to get correct orientation
    return best_score, ''.join(reversed(aligned_v)), ''.join(reversed(aligned_w))


# Example usage:
score, a1, a2 = smith_waterman_affine(
    v="ACACTG",
    w="ATCTG",
    match=2,
    mismatch=1,
    gap_open=3,
    gap_extend=1
)

print("Best Local Score:", score)
print(a1)
print(a2)

Best Local Score: 6
CTG
CTG


In [4]:
import numpy as np

def smith_waterman_affine_numpy(v, w, match, mismatch, gap_open, gap_extend):
    """
    NumPy-optimized Smith-Waterman local alignment with affine gaps.
    Returns: best_score, aligned_v, aligned_w
    """

    def score_matrix(v, w):
        """Create a |v|x|w| matrix of match/mismatch scores."""
        v_arr = np.array(list(v))[:, None]   # column vector
        w_arr = np.array(list(w))[None, :]   # row vector
        return np.where(v_arr == w_arr, match, -mismatch)

    n, m = len(v), len(w)

    # DP matrices
    M  = np.zeros((n+1, m+1), dtype=int)
    Ix = np.zeros((n+1, m+1), dtype=int)
    Iy = np.zeros((n+1, m+1), dtype=int)

    # Backtrack matrices: store (state, i, j) as tuples
    backM  = [[None]*(m+1) for _ in range(n+1)]
    backIx = [[None]*(m+1) for _ in range(n+1)]
    backIy = [[None]*(m+1) for _ in range(n+1)]

    # Score lookup table
    S = score_matrix(v, w)

    best_score = 0
    best_pos = (0, 0)
    best_matrix = "M"

    # Fill dynamic programming matrices
    for i in range(1, n+1):
        # Vectorized candidates for M row
        prev_match = M[i-1, 1:m+1]
        prev_Ix    = Ix[i-1, 1:m+1]
        prev_Iy    = Iy[i-1, 1:m+1]

        match_scores = prev_match + S[i-1]
        gapx_scores  = prev_Ix    + S[i-1]
        gapy_scores  = prev_Iy    + S[i-1]

        # Compute M row
        row_M = np.maximum.reduce([match_scores, gapx_scores, gapy_scores, np.zeros(m)])
        M[i, 1:m+1] = row_M

        # Compute Ix row
        open_gap    = M[i-1, 1:m+1] - (gap_open + gap_extend)
        extend_gap  = Ix[i-1, 1:m+1] - gap_extend
        row_Ix = np.maximum.reduce([open_gap, extend_gap, np.zeros(m)])
        Ix[i, 1:m+1] = row_Ix

        # Compute Iy row (requires shift)
        open_gap     = M[i, 0:m] - (gap_open + gap_extend)
        extend_gap   = Iy[i, 0:m] - gap_extend
        row_Iy = np.maximum.reduce([open_gap, extend_gap, np.zeros(m)])
        Iy[i, 1:m+1] = row_Iy

        # Track best in this row
        row_candidates = np.maximum.reduce([row_M, row_Ix, row_Iy])
        idx = np.argmax(row_candidates)
        row_best = row_candidates[idx]
        if row_best > best_score:
            best_score = row_best
            best_pos = (i, idx+1)
            # Determine which matrix the max came from
            if row_best == row_M[idx]:
                best_matrix = "M"
            elif row_best == row_Ix[idx]:
                best_matrix = "Ix"
            else:
                best_matrix = "Iy"

        # Fill backtracking (scalar because backtracking is non-vector)
        for j in range(1, m+1):

            # backtrack for M
            cand = [
                (M[i-1][j-1], "M", i-1, j-1),
                (Ix[i-1][j-1], "Ix", i-1, j-1),
                (Iy[i-1][j-1], "Iy", i-1, j-1)
            ]
            bestprev = max(cand, key=lambda x: x[0])
            if M[i][j] == 0:
                backM[i][j] = None
            else:
                backM[i][j] = (bestprev[1], bestprev[2], bestprev[3])

            # backtrack for Ix
            if Ix[i][j] == 0:
                backIx[i][j] = None
            elif M[i-1][j] - (gap_open + gap_extend) >= Ix[i-1][j] - gap_extend:
                backIx[i][j] = ("M", i-1, j)
            else:
                backIx[i][j] = ("Ix", i-1, j)

            # backtrack for Iy
            if Iy[i][j] == 0:
                backIy[i][j] = None
            elif M[i][j-1] - (gap_open + gap_extend) >= Iy[i][j-1] - gap_extend:
                backIy[i][j] = ("M", i, j-1)
            else:
                backIy[i][j] = ("Iy", i, j-1)

    # Backtracking to recover alignment
    i, j = best_pos
    matrix = best_matrix
    aligned_v = []
    aligned_w = []

    while True:
        if matrix == "M":
            if M[i][j] == 0:
                break
            source = backM[i][j]
            aligned_v.append(v[i-1])
            aligned_w.append(w[j-1])

        elif matrix == "Ix":
            if Ix[i][j] == 0:
                break
            source = backIx[i][j]
            aligned_v.append(v[i-1])
            aligned_w.append("-")

        elif matrix == "Iy":
            if Iy[i][j] == 0:
                break
            source = backIy[i][j]
            aligned_v.append("-")
            aligned_w.append(w[j-1])

        if source is None:
            break

        matrix, i, j = source

    return best_score, ''.join(reversed(aligned_v)), ''.join(reversed(aligned_w))


# Example usage:
score, a1, a2 = smith_waterman_affine_numpy(
    "ACACTG", "ATCTG",
    match=2, mismatch=1,
    gap_open=3, gap_extend=1
)

print("Best local score:", score)
print(a1)
print(a2)

Best local score: 3.0
A
A


In [9]:
def overlap_alignment(v, w, match, mismatch, indel):
    """
    Solve the Overlap Alignment Problem.
    v, w: nucleotide strings
    match: match reward (positive)
    mismatch: mismatch penalty (positive)
    indel: indel penalty (positive)
    """

    # Scoring helper
    def s(a, b):
        return match if a == b else -mismatch

    n, m = len(v), len(w)

    # DP and backtrack
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    back = [[None] * (m + 1) for _ in range(n + 1)]

    # Initialization:
    # First row = 0 (w can start anywhere)
    for j in range(1, m + 1):
        dp[0][j] = 0
        back[0][j] = None

    # First column incurs normal indel penalties,
    # because v must end at the end (suffix).
    for i in range(1, n + 1):
        dp[i][0] = dp[i - 1][0] - indel
        back[i][0] = (i - 1, 0)

    # Fill DP matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match_score = dp[i - 1][j - 1] + s(v[i - 1], w[j - 1])
            delete_score = dp[i - 1][j] - indel
            insert_score = dp[i][j - 1] - indel

            best = max(match_score, delete_score, insert_score)
            dp[i][j] = best

            if best == match_score:
                back[i][j] = (i - 1, j - 1)
            elif best == delete_score:
                back[i][j] = (i - 1, j)
            else:
                back[i][j] = (i, j - 1)

    # For overlap alignment:
    # pick best cell in LAST ROW (i = n), any column j
    best_j = max(range(m + 1), key=lambda j: dp[n][j])
    best_score = dp[n][best_j]

    # Backtrack from (n, best_j)
    i, j = n, best_j
    aligned_v = []
    aligned_w = []

    while i > 0 or j > 0:
        if back[i][j] is None:
            break
        pi, pj = back[i][j]
        if pi == i - 1 and pj == j - 1:
            aligned_v.append(v[i - 1])
            aligned_w.append(w[j - 1])
        elif pi == i - 1:
            aligned_v.append(v[i - 1])
            aligned_w.append('-')
        else:
            aligned_v.append('-')
            aligned_w.append(w[j - 1])
        i, j = pi, pj

    return best_score, ''.join(reversed(aligned_v)), ''.join(reversed(aligned_w))


# Example:
v = "TGTTACTCGACAGTAGCTTATAGTTAACTTAGCCGCCAGCTCCCGCGCCGAATCAAATATAACATCCGTATAGAACTTTCTTTGTTGGCCGTATACGCTTAAACTCGGTATCTACAACGTTGGTCGCACATCTATAACTGGTGTTCGTAGCAAGTGTTACACTAATGGTATCTGAAACACTCGCCGTAATCTACAAACGGGGTCCAAACAGTGACCAAGTAATCACGGATTATCTCGGATCAGCTGGCTGAAACTATTGCGCCAATCAATCGACACCTCTCCACCAGTCGGTCCCTCGTGCTAGAATATACGACAGCACATTTATGGCTTAGCTACTACCCGCTACACCATCTTTTGGACCTGGTACTCTTAGTATAAATGGAGCGGTGGTTTGGTGCGTTCGCAACGTACAGCAAAGCGTTGTTCCGCTGCATTGCACAAATAACAATCGGCCCATGGGAAGCTCCTAAAATCAGGGACTATTAGAATCCCCATTGCACTGCAGGGGTCCCGTTGCTATGGCTGAGGTGGAAGGTATCGGGGTGGCAACGGCTAGGCACACTGGAGATGGACGTTAGGACGGAATGAGGACGACAGTTGCTCTCGACGAGACCATAGTGTGATAGTGCGAGCGATTACCTGGAGCCTTGGCGCATCCCCGTTCAGTTAGCCGTAAGGGTGGAACCTGAGGGAAGTATAGGATCGCATCATCCAACGCTTGGTTCGTCGACGGGCCGACCCAAAAACTGACGAGAAGCAGTTCGAAGAATGGACCATGCGGGACATGTAAGATCTGTAATACTCTCCAAGCCACAACCTGGTAACCCAGGTGACCGCACGCAGCTAATAAAATGCCCTGCCTCAGCCTGTCTGAACAACTCCGGAAACGTGGCGCTCTCT"
w = "TGTTACTCGACAGTAGTTTATAGTTAGTTCTTGCCAGCCAGCCCCTGCGCGCCCTACCGGGTGATCAAATAACCAAACATCCGTATAGAACTTTCTTTGTTGGCCGACCCTTCAACTCCGTATCAACAACGTTGGTCACACATCTAGCTAACTGGTTAGCTAGAACCAGTGCGGCGATGGTATCTGTCGCCGTAATCTAAATTGTTCTCCAAACAGAGTCCAAGGAATCAAGCGGATCAGGTGGCTGAAATTATTGCGCGTGAGACGGAATTCGACACCTCTCCACCAGTCGGTCCCTCGTTGACTCGGCTAGAATATACGACAGCACATCTATTGCTCTTTAGTATCGCTACACCTGCGCTACCTGGTACTCTTTAAAGTATATATGGAGCGGTGGTTTGGTCGCTAAACACGTACAGCATACTGAAGAACAGCTAAAAGCCTACTAGTAGCGACTATTCCATTGCACTTCCGGCTGGTGGAAGGTATCGGAAGTGGCAACACATTGTGGAACACGGGAGATGGACAGGGAGACAGTTTCGGAGTGTTCCTTAAAGATAGTGCGAGCGATTACCTGGAAAAAAATCCCTTGCGCACGGACCCCGTTCAGATAGCCGTAATCCATGGTGGAAGGGACAAGAGTTCAAAATAGGACCGCATCATCCAGGACGGGCCGACCCAAAGAAACTGACGAGAAGCAGTTCGAACAATGTTGTCGAACCATTACCGGGACAAGTATAAGATCTTGAATACTCCAGCGTTTGGGAACTCAGGTGACGCCTGCGCACGCCTGACGCAGACCCTCCAATGTCCTGCCTCTGCCTGTCTGCAGGAAACGTGGCAATTCTCT"

score, a_v, a_w = overlap_alignment(v, w, match=1, mismatch=1, indel=2)
print(score)
print(a_v)
print(a_w)

163
TGTTACTCGACAGTAGCTTATAGTTAACT-TAGCC-GCCAGCTCC--CGCG-----CC--G-AATCAAAT-A--TAACATCCGTATAGAACTTTCTTTGTTGGCCGTATACGCTTAAACTCGGTATCTACAACGTTGGTCGCACATCTA--TAACTGGTGTTCGTAGCAAGTGTTACACTAATGGTATCTGAAACACTCGCCGTAATCTACAAACGGGGTCCAAACAGTGACCAAGTAATCACGGATTATCTCGGATCAGCTGGCTGAAACTATTGCGC--CA-A-TCAA-TCGACACCTCTCCACCAGTCGGTCCCTC---G--T--GCTAGAATATACGACAGCACATTTATGGCTTAGCTACTACCCGCTACACCATCTTTTGGACCTGGTACTC-TT--AGTATAAATGGAGCGGTGGTTTGGTGCGTTCGCAACGTACAGCAAAGCGTTGTTCCGCTGCATTGCACAAATAACAATCGGCCCATGGGAAGCTCCTAAAATCAGGGACTATTAGAATCCCCATTGCACTGCAGGGGTCCCGTTGCTATGGCTGAGGTGGAAGGTATCGG-GGTGGCAACGGCTAGGCACACTGGAGATGGACGTTAGGACGGAATGAGGACGACAGTTGCTCTCGACGAG-ACCATAGTGTGATAGTGCGAGCGATTACCTGG------A-GCCTTG-GCGC-ATCCCCGTTCAGTTAGCCGTAAGGGTGGAACCTGAGGGAA-GTATAGGATCGCATCAT-CCAACGCTTGGTTCGTCGACGGGCCGACCC-AA-AAACTGACGAGAAGCAGTTCGAAGAA---TG--G-ACCA-T-GCGGGAC-A-TGTAAGATCTGTAATACT-CTCCAAGCCACAAC-CTGGTAAC-CCAG-G-TGACCGCACGCAGCTAATAAAATGCCCTGCCTCAGCCTGTCTGAACAACTCCGGAAACGTGGC-GCTCTCT
TGTTACTCGACAGTAGTTTATAGTTAGTTCTT

In [11]:
import numpy as np

def overlap_alignment_numpy(v, w, match, mismatch, indel):
    """
    NumPy-optimized Overlap Alignment
    (align suffix of v to prefix of w).

    match: match reward (positive)
    mismatch: mismatch penalty (positive)
    indel: indel penalty (positive)
    """
    n, m = len(v), len(w)

    # Precompute match/mismatch scoring table (1D for w)
    w_chars = np.array(list(w))

    def score_vec(char):
        """Vectorized scoring: v[i] vs all w[j]."""
        return np.where(w_chars == char, match, -mismatch)

    # DP arrays for rows
    prev = np.zeros(m + 1, dtype=int)
    curr = np.zeros(m + 1, dtype=int)

    # Backtracking matrix (store directions)
    # 0 = none/start, 1 = diag, 2 = up, 3 = left
    backtrack = np.zeros((n + 1, m + 1), dtype=np.int8)

    # Initialize first column: normal because v-suffix must end at end of DP
    for i in range(1, n + 1):
        prev[0] = prev[0] - indel
        backtrack[i][0] = 2   # from up

    # First row stays zero (overlap alignment allows w-prefix)
    # prev is already zeros for j > 0, and stays that way.

    # Fill DP row-by-row
    for i in range(1, n + 1):
        curr[0] = prev[0] - indel  # normal deletion in first column
        backtrack[i][0] = 2        # up

        row_score = score_vec(v[i-1])

        diag = prev[:-1] + row_score
        up = prev[1:] - indel
        left = curr[:-1] - indel

        # Compute max of three
        maxvals = np.maximum(diag, np.maximum(up, left))

        curr[1:] = maxvals

        # Fill backtracking
        for j in range(1, m + 1):
            d = diag[j-1]
            u = up[j-1]
            l = left[j-1]

            if curr[j] == d:
                backtrack[i][j] = 1  # diag
            elif curr[j] == u:
                backtrack[i][j] = 2  # up
            else:
                backtrack[i][j] = 3  # left

        # Move to next iteration
        prev, curr = curr, prev   # swap for next row

    # prev currently holds final row dp[n][*]
    final_dp_row = prev

    # Best overlap ends somewhere in last row: dp[n][j] for any j
    best_j = int(np.argmax(final_dp_row))
    best_score = int(final_dp_row[best_j])

    # Backtrack from (n, best_j)
    aligned_v = []
    aligned_w = []

    i, j = n, best_j

    while i > 0 and j >= 0:
        bt = backtrack[i][j]
        if bt == 1:  # diag
            aligned_v.append(v[i-1])
            aligned_w.append(w[j-1])
            i -= 1
            j -= 1
        elif bt == 2:  # up
            aligned_v.append(v[i-1])
            aligned_w.append('-')
            i -= 1
        elif bt == 3:  # left
            aligned_v.append('-')
            aligned_w.append(w[j-1])
            j -= 1
        else:
            break

    return best_score, ''.join(reversed(aligned_v)), ''.join(reversed(aligned_w))


# Example usage:
v = "TGTTACTCGACAGTAGCTTATAGTTAACTTAGCCGCCAGCTCCCGCGCCGAATCAAATATAACATCCGTATAGAACTTTCTTTGTTGGCCGTATACGCTTAAACTCGGTATCTACAACGTTGGTCGCACATCTATAACTGGTGTTCGTAGCAAGTGTTACACTAATGGTATCTGAAACACTCGCCGTAATCTACAAACGGGGTCCAAACAGTGACCAAGTAATCACGGATTATCTCGGATCAGCTGGCTGAAACTATTGCGCCAATCAATCGACACCTCTCCACCAGTCGGTCCCTCGTGCTAGAATATACGACAGCACATTTATGGCTTAGCTACTACCCGCTACACCATCTTTTGGACCTGGTACTCTTAGTATAAATGGAGCGGTGGTTTGGTGCGTTCGCAACGTACAGCAAAGCGTTGTTCCGCTGCATTGCACAAATAACAATCGGCCCATGGGAAGCTCCTAAAATCAGGGACTATTAGAATCCCCATTGCACTGCAGGGGTCCCGTTGCTATGGCTGAGGTGGAAGGTATCGGGGTGGCAACGGCTAGGCACACTGGAGATGGACGTTAGGACGGAATGAGGACGACAGTTGCTCTCGACGAGACCATAGTGTGATAGTGCGAGCGATTACCTGGAGCCTTGGCGCATCCCCGTTCAGTTAGCCGTAAGGGTGGAACCTGAGGGAAGTATAGGATCGCATCATCCAACGCTTGGTTCGTCGACGGGCCGACCCAAAAACTGACGAGAAGCAGTTCGAAGAATGGACCATGCGGGACATGTAAGATCTGTAATACTCTCCAAGCCACAACCTGGTAACCCAGGTGACCGCACGCAGCTAATAAAATGCCCTGCCTCAGCCTGTCTGAACAACTCCGGAAACGTGGCGCTCTCT"
w = "TGTTACTCGACAGTAGTTTATAGTTAGTTCTTGCCAGCCAGCCCCTGCGCGCCCTACCGGGTGATCAAATAACCAAACATCCGTATAGAACTTTCTTTGTTGGCCGACCCTTCAACTCCGTATCAACAACGTTGGTCACACATCTAGCTAACTGGTTAGCTAGAACCAGTGCGGCGATGGTATCTGTCGCCGTAATCTAAATTGTTCTCCAAACAGAGTCCAAGGAATCAAGCGGATCAGGTGGCTGAAATTATTGCGCGTGAGACGGAATTCGACACCTCTCCACCAGTCGGTCCCTCGTTGACTCGGCTAGAATATACGACAGCACATCTATTGCTCTTTAGTATCGCTACACCTGCGCTACCTGGTACTCTTTAAAGTATATATGGAGCGGTGGTTTGGTCGCTAAACACGTACAGCATACTGAAGAACAGCTAAAAGCCTACTAGTAGCGACTATTCCATTGCACTTCCGGCTGGTGGAAGGTATCGGAAGTGGCAACACATTGTGGAACACGGGAGATGGACAGGGAGACAGTTTCGGAGTGTTCCTTAAAGATAGTGCGAGCGATTACCTGGAAAAAAATCCCTTGCGCACGGACCCCGTTCAGATAGCCGTAATCCATGGTGGAAGGGACAAGAGTTCAAAATAGGACCGCATCATCCAGGACGGGCCGACCCAAAGAAACTGACGAGAAGCAGTTCGAACAATGTTGTCGAACCATTACCGGGACAAGTATAAGATCTTGAATACTCCAGCGTTTGGGAACTCAGGTGACGCCTGCGCACGCCTGACGCAGACCCTCCAATGTCCTGCCTCTGCCTGTCTGCAGGAAACGTGGCAATTCTCT"

score, av, aw = overlap_alignment_numpy(
    v, w, match=1, mismatch=1, indel=1
)

print("Best score:", score)
print(av)
print(aw)


Best score: 176
TGTTACTCGACAG-TAGCTT-ATAGTT-AACTTAGCCGCCAGCTCCCGCGCCG-AATCAAATATAACATCCGTATAGAACTTTCTTTGTTGGCCGTATA-CGCTTAAACTCGGTATCTACAACGTTGGTCGCACATCTATA-ACTGG-TGTTCGTAGCAAGTG-TTACACTAATGGTA-TCT-GAAACACTCGCCGTAATCTACAAACGGGGTCCAAACAGTGACCAAGTAATCACG-GATTATCTCGGATCAGCTGGCTGAAACTATTGCGCCA-ATCA-ATCGACACC-TCTCCACCAGT-CGGTCCCT-CGTGCTA-GAATATACGAC-AGCACATTT-ATGGCTTAGCT-ACTACCCGCTACACCATCTTTTGGA-CCTGG-TACT-CT-TAGTA-TAAATG-GAGCGGTGG-TTTGGTGCGTTCG-CAACGTACAGC-AAAGCGTTGTTCCGCTGCATTGCACAA-ATAACAAT-CGG-CCCATG-GGAAGCTCCTAAAATCAG-GG-ACTATTAGAATC-CCCATTGCACTGCAGGGGTCCCGTTGCTA-TGGCTGAGGTGGAAGGTATCGGGG-TG-GCA-ACGGCTAGG-CACACTGGA-GAT-G-GACGTTAGGACGGAATGAGGACGACAGTTGCTCTC-GACGAGACCATAGTGT-GATAGTGCGAGCGATTACCTGGAGC-CT-TG-GC-GC-ATCCCCGTTCAGTT-AGCCGTAAG-GGTGGAACCTG-AG-GG-AAGTATAGGA-TCGCATCATCCAAC-GCT-TGGTT-CGTC-GACGG-GC-CGAC-CCAAAAACTGACGAGA-AGCAGTTCGAAGAATGGACCA-TGCGG-GACAT-GT-AAGATCTGTAATACTCT-CCAAGCC-ACAACCTGGTAACCC-AGGTGAC-CGCA-CGCAGCTA-ATAAAA-TGCCCTGCCTCAGCC-TGTCTGAACA-ACTCCGGAAA-CGTGG-CGCT-CT-CT
A

In [8]:
def overlap_alignment(match, mismatch, indel, v, w):
    n, m = len(v), len(w)
    # Initialize DP table
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    backtrack = [[None] * (m + 1) for _ in range(n + 1)]

    # Fill DP table
    for i in range(1, n + 1):
        dp[i][0] = -i * indel
        backtrack[i][0] = "up"
    for j in range(1, m + 1):
        dp[0][j] = -j * indel
        backtrack[0][j] = "left"

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            score_sub = dp[i-1][j-1] + (match if v[i-1] == w[j-1] else -mismatch)
            score_del = dp[i-1][j] - indel
            score_ins = dp[i][j-1] - indel
            dp[i][j] = max(score_sub, score_del, score_ins)

            if dp[i][j] == score_sub:
                backtrack[i][j] = "diag"
            elif dp[i][j] == score_del:
                backtrack[i][j] = "up"
            else:
                backtrack[i][j] = "left"

    # Find best overlap: suffix of v (last row) vs prefix of w (last col)
    max_score = float("-inf")
    max_pos = (0, 0)

    # last row
    for j in range(m + 1):
        if dp[n][j] > max_score:
            max_score = dp[n][j]
            max_pos = (n, j)

    # last column
    for i in range(n + 1):
        if dp[i][m] > max_score:
            max_score = dp[i][m]
            max_pos = (i, m)

    # Traceback
    i, j = max_pos
    v_aln, w_aln = "", ""
    while i > 0 and j > 0:
        if backtrack[i][j] == "diag":
            v_aln = v[i-1] + v_aln
            w_aln = w[j-1] + w_aln
            i -= 1
            j -= 1
        elif backtrack[i][j] == "up":
            v_aln = v[i-1] + v_aln
            w_aln = "-" + w_aln
            i -= 1
        elif backtrack[i][j] == "left":
            v_aln = "-" + v_aln
            w_aln = w[j-1] + w_aln
            j -= 1
        else:
            break

    return max_score, v_aln, w_aln


# --- Test with Sample Input ---
match, mismatch, indel = 1, 1, 2
v = "TGTTACTCGACAGTAGCTTATAGTTAACTTAGCCGCCAGCTCCCGCGCCGAATCAAATATAACATCCGTATAGAACTTTCTTTGTTGGCCGTATACGCTTAAACTCGGTATCTACAACGTTGGTCGCACATCTATAACTGGTGTTCGTAGCAAGTGTTACACTAATGGTATCTGAAACACTCGCCGTAATCTACAAACGGGGTCCAAACAGTGACCAAGTAATCACGGATTATCTCGGATCAGCTGGCTGAAACTATTGCGCCAATCAATCGACACCTCTCCACCAGTCGGTCCCTCGTGCTAGAATATACGACAGCACATTTATGGCTTAGCTACTACCCGCTACACCATCTTTTGGACCTGGTACTCTTAGTATAAATGGAGCGGTGGTTTGGTGCGTTCGCAACGTACAGCAAAGCGTTGTTCCGCTGCATTGCACAAATAACAATCGGCCCATGGGAAGCTCCTAAAATCAGGGACTATTAGAATCCCCATTGCACTGCAGGGGTCCCGTTGCTATGGCTGAGGTGGAAGGTATCGGGGTGGCAACGGCTAGGCACACTGGAGATGGACGTTAGGACGGAATGAGGACGACAGTTGCTCTCGACGAGACCATAGTGTGATAGTGCGAGCGATTACCTGGAGCCTTGGCGCATCCCCGTTCAGTTAGCCGTAAGGGTGGAACCTGAGGGAAGTATAGGATCGCATCATCCAACGCTTGGTTCGTCGACGGGCCGACCCAAAAACTGACGAGAAGCAGTTCGAAGAATGGACCATGCGGGACATGTAAGATCTGTAATACTCTCCAAGCCACAACCTGGTAACCCAGGTGACCGCACGCAGCTAATAAAATGCCCTGCCTCAGCCTGTCTGAACAACTCCGGAAACGTGGCGCTCTCT"
w = "TGTTACTCGACAGTAGTTTATAGTTAGTTCTTGCCAGCCAGCCCCTGCGCGCCCTACCGGGTGATCAAATAACCAAACATCCGTATAGAACTTTCTTTGTTGGCCGACCCTTCAACTCCGTATCAACAACGTTGGTCACACATCTAGCTAACTGGTTAGCTAGAACCAGTGCGGCGATGGTATCTGTCGCCGTAATCTAAATTGTTCTCCAAACAGAGTCCAAGGAATCAAGCGGATCAGGTGGCTGAAATTATTGCGCGTGAGACGGAATTCGACACCTCTCCACCAGTCGGTCCCTCGTTGACTCGGCTAGAATATACGACAGCACATCTATTGCTCTTTAGTATCGCTACACCTGCGCTACCTGGTACTCTTTAAAGTATATATGGAGCGGTGGTTTGGTCGCTAAACACGTACAGCATACTGAAGAACAGCTAAAAGCCTACTAGTAGCGACTATTCCATTGCACTTCCGGCTGGTGGAAGGTATCGGAAGTGGCAACACATTGTGGAACACGGGAGATGGACAGGGAGACAGTTTCGGAGTGTTCCTTAAAGATAGTGCGAGCGATTACCTGGAAAAAAATCCCTTGCGCACGGACCCCGTTCAGATAGCCGTAATCCATGGTGGAAGGGACAAGAGTTCAAAATAGGACCGCATCATCCAGGACGGGCCGACCCAAAGAAACTGACGAGAAGCAGTTCGAACAATGTTGTCGAACCATTACCGGGACAAGTATAAGATCTTGAATACTCCAGCGTTTGGGAACTCAGGTGACGCCTGCGCACGCCTGACGCAGACCCTCCAATGTCCTGCCTCTGCCTGTCTGCAGGAAACGTGGCAATTCTCT"

score, v_aln, w_aln = overlap_alignment(match, mismatch, indel, v, w)
print(score)
print(v_aln)
print(w_aln)

163
TGTTACTCGACAGTAGCTTATAGTTAACT-TAGCC-GCCAGCTCC--CGCG-----CC--G-AATCAAAT-A--TAACATCCGTATAGAACTTTCTTTGTTGGCCGTATACGCTTAAACTCGGTATCTACAACGTTGGTCGCACATCTA--TAACTGGTGTTCGTAGCAAGTGTTACACTAATGGTATCTGAAACACTCGCCGTAATCTACAAACGGGGTCCAAACAGTGACCAAGTAATCACGGATTATCTCGGATCAGCTGGCTGAAACTATTGCGC--CA-A-TCAA-TCGACACCTCTCCACCAGTCGGTCCCTC---G--T--GCTAGAATATACGACAGCACATTTATGGCTTAGCTACTACCCGCTACACCATCTTTTGGACCTGGTACTC-TT--AGTATAAATGGAGCGGTGGTTTGGTGCGTTCGCAACGTACAGCAAAGCGTTGTTCCGCTGCATTGCACAAATAACAATCGGCCCATGGGAAGCTCCTAAAATCAGGGACTATTAGAATCCCCATTGCACTGCAGGGGTCCCGTTGCTATGGCTGAGGTGGAAGGTATCGG-GGTGGCAACGGCTAGGCACACTGGAGATGGACGTTAGGACGGAATGAGGACGACAGTTGCTCTCGACGAG-ACCATAGTGTGATAGTGCGAGCGATTACCTGG------A-GCCTTG-GCGC-ATCCCCGTTCAGTTAGCCGTAAGGGTGGAACCTGAGGGAA-GTATAGGATCGCATCAT-CCAACGCTTGGTTCGTCGACGGGCCGACCC-AA-AAACTGACGAGAAGCAGTTCGAAGAA---TG--G-ACCA-T-GCGGGAC-A-TGTAAGATCTGTAATACT-CTCCAAGCCACAAC-CTGGTAAC-CCAG-G-TGACCGCACGCAGCTAATAAAATGCCCTGCCTCAGCCTGTCTGAACAACTCCGGAAACGTGGC-GCTCTCT
TGTTACTCGACAGTAGTTTATAGTTAGTTCTT

In [14]:
def middle_edge_linear_space(v, w, match, mismatch, indel):
    """
    Compute the middle edge in the alignment graph between v and w.
    Uses global alignment with linear‑gap penalties.

    v, w: strings (DNA, RNA, or protein)
    match: reward
    mismatch: penalty
    indel: penalty
    """

    def s(a, b):
        return match if a == b else -mismatch

    n, m = len(v), len(w)
    mid = m // 2

    # -------- Forward DP until middle column --------
    prev = [0] * (n + 1)
    for i in range(1, n + 1):
        prev[i] = prev[i - 1] - indel

    # Compute dp column-by-column up to mid
    for j in range(1, mid + 1):
        curr = [0] * (n + 1)
        curr[0] = -j * indel
        for i in range(1, n + 1):
            match_val = prev[i - 1] + s(v[i - 1], w[j - 1])
            delete_val = prev[i] - indel
            insert_val = curr[i - 1] - indel
            curr[i] = max(match_val, delete_val, insert_val)
        prev = curr

    forward_col = prev  # DP column at j = mid

    # -------- Backward DP from right side until middle column --------
    v_rev = v[::-1]
    w_rev = w[::-1]

    prev = [0] * (n + 1)
    for i in range(1, n + 1):
        prev[i] = prev[i - 1] - indel

    # Compute dp for reversed strings up to column (m - mid)
    steps = m - mid
    for j in range(1, steps + 1):
        curr = [0] * (n + 1)
        curr[0] = -j * indel
        for i in range(1, n + 1):
            match_val = prev[i - 1] + s(v_rev[i - 1], w_rev[j - 1])
            delete_val = prev[i] - indel
            insert_val = curr[i - 1] - indel
            curr[i] = max(match_val, delete_val, insert_val)
        prev = curr

    backward_col = prev[::-1]   # reverse to align rows

    # -------- Identify middle row --------
    best_i = max(range(n + 1), key=lambda i: forward_col[i] + backward_col[i])

    # -------- Determine outgoing middle edge --------
    # We compare three possibilities:
    # (best_i, mid) -> (best_i+1, mid)     deletion
    # (best_i, mid) -> (best_i, mid+1)     insertion
    # (best_i, mid) -> (best_i+1, mid+1)   match/mismatch

    # Compute score contributions
    node_score = forward_col[best_i]

    # Option 1: down (delete)
    if best_i < n:
        score_down = node_score - indel + backward_col[best_i + 1]
    else:
        score_down = float('-inf')

    # Option 2: right (insert)
    if mid < m:
        score_right = node_score - indel + backward_col[best_i]
    else:
        score_right = float('-inf')

    # Option 3: diag (match/mismatch)
    if best_i < n and mid < m:
        score_diag = node_score + s(v[best_i], w[mid]) + backward_col[best_i + 1]
    else:
        score_diag = float('-inf')

    # Choose direction
    best_move = max(
        [(score_down, (best_i + 1, mid)),
         (score_right, (best_i, mid + 1)),
         (score_diag, (best_i + 1, mid + 1))],
        key=lambda x: x[0]
    )

    next_i, next_j = best_move[1]

    return (best_i, mid), (next_i, next_j)


# ---------------- EXAMPLE ----------------
v = "CATTCAATGTGGGCTTGATATTGTAGACCGGCAATCTAGCGTACGCATACTTCCAAGCTGAGGCCACTCATTGGGAACATCAGTCGCTACGTCTAGGGTATTGCGCCGTATGAGGTGTGTTAAGTTATTAACAACTGACCAGACTTTAAGCCCACGCAGGGAGGGTGAGTAGCATCACATCCCGCCTCAGCCGGGCCGCTAGAGGTTTTCTGGGTGGCTAATGAAATTCGGCGACTTTAGGGATAACCTTCTCCTTCCCACAACCCATTTTCGCCCGAGGGCTCACACAAATGGCCCTGTCGTAACATAAGTAGGGGAATGATCTCTCAAATTGATTGAAGAGGGAGTCGAGTAGCAAAGATGCTTGATAACCTGTTTGGAGCACAACATCGAATGGGGAGCTATAGTCAGGTGCATTCCTCCGCAGGTATAGCGTACTCGTTATTGACATCCTACACTCAGCATCAGCATTTAGGTAATTTCGCAGGTCCATTGCGGGGGGGTTAGCGGCACCGGGAACTATTAAATGGTGATATCACTGATACGAGTGACACACCCCCGAGTTTAGAGCGGCTCATCGTTCTCCTCCGCTCACTTCGAAGTCTGCACTTACGGAAGGGGCGATGCATAGGAGCGATGAACCGGAATTGAGCCCCCTCAAGATTTACCCACGAGATGATCCGTTGCGCGCAGGGGTAGAGGGACGTTAGCGACTATAATTAGTTCCCCTCCCACGATTTGAAACAGCCGCACGATCACCCGGAACGTGCAAGAAGAGCCCCAACTCTAGCGGATCTAAGTAGAGGGGACGATTCCTAAGGCGTTTCCCTTGCTGGAGATGCGGAAGGAGGGAACCTTATCGCTGGCGAATCTTAAGTTCGAGTCTAAGGTACTACGTCT"
w = "CATTCTTGATATTGTACATACAGTCGCTCCTTAAGTACGCAGGCAAGTCTGCAGGATGAGGCCATCCTTGGGAACATCAGTCTGTACGACAGAACTTGTCGTAGGAGGTGTGTTATCAACACAGCAGCCCTAACGGCACGCACCATCTGTGCAGTCGCACACATGGCCACATTGCCTTCAGCCGGGCCGCATGCAAGACAGGCCAGTTGGCGTCGTGTGGCTACGGGATCGGTAGTTATTACGGCGACATTAGTTCTCCTTCCCATTAGGGCTCACACATGCAAGGGCCCGGATGTAAGTAGGGGACAAATTTGTGATCTCCCAAAATGATTGAATAGGGAGTCGAGTTGCCTTGATAAGAGCTGTTTGGAGCACAACATTGAATGGGGAGCTCGCCAATCGCATGGTTCGCATCATGGTCTTGCGGACACCTATAGTCGCTACAGTCCGACAGCTTCAGCGACCCCATTTGTAGGTAATTTCAGGTCCAACTTGTTTGCGTGGCGGCACCAGGAGATAGTGACACCAGGAGACGAGGGACTCACCCCACGAGTTTAGAGCCTGTCTAAGGCTCAGCCCTCTGACGCTCACTTCGAAGTCCTGCACATACGCCAGATGTCTTTGGATGGACCGGAAGGAGCCGAGGTCGAGAACGCTAGGGCGCGCAGATTCGTGAGGTAGAGGGACGGTAGAGACTATAATTAGTTCCCCTAGACCAACGATTGGAAACACCCCACAGCCCTGCATAGAAGTTTTTCGACGTGCAAGAAGAGCCCCTAGCCAGCGAGTGTAGAGGGTACGATTACTAAGGCGTTTCCCTTTAGATGCAGAAGGGCGGGGAACCTTATCGCTGGCGGATCGGTAGTGCTACCATCTAAGCTGTCTTTGTACTACGTGT"

middle_edge = middle_edge_linear_space(v, w, match=1, mismatch=1, indel=5)
print("Middle Edge:", middle_edge)

Middle Edge: ((457, 449), (457, 450))


In [15]:
def linear_space_alignment(v, w, match, mismatch, indel):
    """
    Compute full global alignment of v and w using Hirschberg's algorithm.
    Requires a MiddleEdge function (provided below).
    """

    def s(a, b):
        return match if a == b else -mismatch

    # -------------------------------------------------------
    #   Forward DP to a given right boundary (linear space)
    # -------------------------------------------------------
    def compute_forward(v, w):
        n, m = len(v), len(w)
        prev = [0] * (n + 1)
        for i in range(1, n + 1):
            prev[i] = prev[i - 1] - indel

        for j in range(1, m + 1):
            curr = [0] * (n + 1)
            curr[0] = -j * indel
            for i in range(1, n + 1):
                curr[i] = max(
                    prev[i] - indel,          # deletion
                    curr[i - 1] - indel,      # insertion
                    prev[i - 1] + s(v[i-1], w[j-1])  # match/mismatch
                )
            prev = curr
        return prev

    # -------------------------------------------------------
    #   Middle Edge computation (Hirschberg forward + backward)
    # -------------------------------------------------------
    def middle_edge(v, w, top, bottom, left, right):
        """
        Returns ((i, j), (i2, j2), direction)
        direction is one of: "→", "↓", "↘".
        """

        v_sub = v[top:bottom]
        w_sub = w[left:right]
        n, m = len(v_sub), len(w_sub)
        mid = m // 2

        # Forward DP up to mid column
        forward = compute_forward(v_sub, w_sub[:mid])

        # Backwards DP from right side
        backward = compute_forward(v_sub[::-1], w_sub[mid:][::-1])
        backward = backward[::-1]

        # Find best row
        best_i = max(range(n + 1), key=lambda i: forward[i] + backward[i])

        # Middle column in full coordinates
        mid_col = left + mid
        mid_row = top + best_i

        # Determine direction (which next step is optimal)
        candidates = []

        # Diagonal
        if best_i < n and mid < m:
            diag = forward[best_i] + s(v_sub[best_i], w_sub[mid]) + backward[best_i+1]
            candidates.append((diag, "↘"))

        # Down
        if best_i < n:
            down = forward[best_i] - indel + backward[best_i+1]
            candidates.append((down, "↓"))

        # Right
        if mid < m:
            right_score = forward[best_i] - indel + backward[best_i]
            candidates.append((right_score, "→"))

        _, direction = max(candidates)

        # Convert direction to next node
        if direction == "↘":
            return (mid_row, mid_col), (mid_row+1, mid_col+1), direction
        elif direction == "↓":
            return (mid_row, mid_col), (mid_row+1, mid_col), direction
        else:
            return (mid_row, mid_col), (mid_row, mid_col+1), direction

    # -------------------------------------------------------
    #   Recursive LinearSpaceAlignment
    # -------------------------------------------------------
    alignment_v = []
    alignment_w = []

    def recurse(top, bottom, left, right):
        # Base cases

        if left == right:
            # Only vertical edges remain
            for i in range(top, bottom):
                alignment_v.append(v[i])
                alignment_w.append("-")
            return

        if top == bottom:
            # Only horizontal edges remain
            for j in range(left, right):
                alignment_v.append("-")
                alignment_w.append(w[j])
            return

        # Recursive case
        mid = (left + right) // 2
        (i1, j1), (i2, j2), direction = middle_edge(v, w, top, bottom, left, right)

        recurse(top, i1, left, mid)

        # output midEdge
        if direction == "↘":
            alignment_v.append(v[i1])
            alignment_w.append(w[j1])
        elif direction == "↓":
            alignment_v.append(v[i1])
            alignment_w.append("-")
        else:  # →
            alignment_v.append("-")
            alignment_w.append(w[j1])

        recurse(i2, bottom, j2, right)

    recurse(0, len(v), 0, len(w))
    return "".join(alignment_v), "".join(alignment_w)

In [16]:
v = "PLEASANTLY"
w = "MEANLY"

a_v, a_w = linear_space_alignment(
    v, w, match=1, mismatch=1, indel=1
)

print(a_v)
print(a_w)

PLEASANTLY
MEA-NL---Y


In [17]:
def hirschberg_alignment(v, w, match, mismatch, indel):
    """
    Returns (score, alignment_v, alignment_w)
    Global alignment using Hirschberg's linear‑space algorithm.
    """

    # ------------------------------------------
    # scoring function
    # ------------------------------------------
    def s(a, b):
        return match if a == b else -mismatch

    # ------------------------------------------
    # Needleman–Wunsch last-column score (linear space)
    # Computes DP for v vs w and returns only the last column
    # ------------------------------------------
    def nw_score(v, w):
        n, m = len(v), len(w)

        prev = [0] * (n + 1)
        for i in range(1, n + 1):
            prev[i] = prev[i - 1] - indel

        for j in range(1, m + 1):
            curr = [0] * (n + 1)
            curr[0] = prev[0] - indel
            for i in range(1, n + 1):
                curr[i] = max(
                    prev[i] - indel,               # delete
                    curr[i - 1] - indel,           # insert
                    prev[i - 1] + s(v[i - 1], w[j - 1])  # match/mismatch
                )
            prev = curr
        return prev

    # ------------------------------------------
    # Hirschberg recursive alignment
    # ------------------------------------------
    def hirschberg(v, w):
        n, m = len(v), len(w)

        # Base cases (small strings)
        if n == 0:
            return ("-" * m, w)
        if m == 0:
            return (v, "-" * n)
        if n == 1 or m == 1:
            return needleman_wunsch_small(v, w)

        mid = m // 2

        # Forward score: v vs w[:mid]
        scoreL = nw_score(v, w[:mid])

        # Backward score: reversed v vs reversed w[mid:]
        scoreR = nw_score(v[::-1], w[mid:][::-1])

        # Best split row index
        k = max(range(n + 1), key=lambda i: scoreL[i] + scoreR[n - i])

        # Recurse
        left_v, left_w = hirschberg(v[:k], w[:mid])
        right_v, right_w = hirschberg(v[k:], w[mid:])

        return (left_v + right_v, left_w + right_w)

    # ------------------------------------------
    # Helper: Needleman–Wunsch for very small substrings
    # ------------------------------------------
    def needleman_wunsch_small(v, w):
        n, m = len(v), len(w)

        dp = [[0] * (m + 1) for _ in range(n + 1)]
        back = [[None] * (m + 1) for _ in range(n + 1)]

        for i in range(1, n + 1):
            dp[i][0] = dp[i - 1][0] - indel
            back[i][0] = "↑"

        for j in range(1, m + 1):
            dp[0][j] = dp[0][j - 1] - indel
            back[0][j] = "←"

        for i in range(1, n + 1):
            for j in range(1, m + 1):
                scores = [
                    (dp[i - 1][j] - indel, "↑"),
                    (dp[i][j - 1] - indel, "←"),
                    (dp[i - 1][j - 1] + s(v[i - 1], w[j - 1]), "↖")
                ]
                dp[i][j], back[i][j] = max(scores, key=lambda x: x[0])

        # backtrack
        i, j = n, m
        av, aw = [], []
        while i > 0 or j > 0:
            if back[i][j] == "↖":
                av.append(v[i - 1])
                aw.append(w[j - 1])
                i -= 1
                j -= 1
            elif back[i][j] == "↑":
                av.append(v[i - 1])
                aw.append("-")
                i -= 1
            else:  # ←
                av.append("-")
                aw.append(w[j - 1])
                j -= 1
        return ("".join(reversed(av)), "".join(reversed(aw)))

    # ------------------------------------------
    # Perform Hirschberg alignment
    # ------------------------------------------
    aln_v, aln_w = hirschberg(v, w)

    # Score the alignment
    score = sum(
        s(a, b) if a != "-" and b != "-" else -indel
        for a, b in zip(aln_v, aln_w)
    )

    return score, aln_v, aln_w

In [20]:
match = 1
mismatch = 1
indel = 5

v = "GCTTCTATCAACGAATCTTCGCATTACATGTAAAACTCAAGCCGGAACCAGATCTCTGGATTAGCGTCGACACGATTTATGCCCAGGTAACTGCATGCTATATTTAGGCGGGGATCCGTAATTCTCGGAAGACAAGAGGCCCTGCTGATTCTGCTATGTCATGTCAGGGTATCGTGCGTCCATTTCATAGGCAATTGTGGTCTAAGATACATCGTGAAATGTCCTCGACGTCCTCCCGTGTGACGTATACAGTTCACGATTTTGCTGGTTCAAGTAGTAATCGTGGCAAAGACGTTGGCGCTTGGATTTTATCGGCACTATTGATCCTCTTGCGGGAAGTGATACCAGAGTGGCGCCAGATGATTGTTATTCGATCGGATAGGGAGCAATCTGCTTTCCTGACCGACCTGAGTTAACTACTATGTTGTCAGGCCAAAGTTGTTTATTGCCTCAGTACTTGTGTGATATAGTCTGATTTGCGATCGTTACGAATTGCATCGACTTTTCGAAAATTGTCACCGAAGGCGCACGCGCGGATACTAACCTTCCCACGGGCTGATGACAGAGGATGACATTACAACTTTCGTATCAGTTCCAGAATTCCTATCGCGCACGTCTGCTCCTTACGGTTGCAATGCGCCAGTTGCTTACGCCCCTTCGCGCATGGCAATCGTTCAAACAATACTGCAAGAGCTCTTATTACAGCTTGAAACATACGAGTAAGTACTGCTGAGGGGCTCCTTGCGTGCCTACTCTGCCGTAATGGGAGAAACAGATCCAATGCATCACTCCAAGTGAGCGCGGCTCACCTGTTTAACGAGTAGGAAAACTTGCACCAGGGATTGCGTAGGTATTTTATCGGCTATCGAATCTCTATAACCAATCCAGCTGCATCCTACGATTTTCTTGTGTGGCAGATCGCCAAGTAGGAGGCATTCGGAGCTGGAAAACTCACGGACTGGGCACCAGGATTCGCTTATCTTGGTCAAACTTGCTATCGAGCTGCCACGACGGAAGTGAAGATAAGTCGGGGGCGTCCGAATGTTCATTGTTGACTTGAAAACAATTGCGAATATCCGCAAGGAAATCTTCTTGAGAGGATAGTATTGTCGATGTCAGTGTCTGTAAAATTTCAGATTATCATAGCCTCGTCCCTGCGGTTTGGGGGTAAGTCCCAGGAAAGTGGAGCCTGCCGGCGCTTGCAGCCAACACTGTTTTCCAGAGGGAAATACCTTGGCTAGAATGTCATTCAGCCGCTGATAAACGCGACTTCAAGGGGACACACCAATCCCTTTTATAACTTCCCTAACATATTAGTACTGAAACAACCAGCCGAGCATTTACTAGAGTTGCGGTGCTCCTAGAGCACAAAGGGGGTTTCGTATTCGCCGGGAATAGTTGTAAGCCGCCTAATAGAAGTACTCGGGAATTTCCGGCGAAGAACTGTATAGGCCTAAGAGGTTTGCTCTGAGGGTTGACGTGTGTGTACATAGATTATCAGGTTACTAGCAATCCACCAGGGGATAGCACATAACTTGGTGGCGTTCCGTGGAGGACGAGGTTCCTCAGCGTCGCGTTGCCCGACTCAACGTATTAGTGAAAAGTTTCGCCCAGACCGGGAAAGTACGAGTTGCCCAATTACGGAACGGGTTTACGTCGGACGGGCCGGGGGCAGTGTAACGACTGGCGGGTATTCAGTGTGGATTGCAACTCTTAGTTGGTCCCTTTTTGATGTACCACGGGCCACACGGCATTCCGGGGATCGGCCTTCTGCAAAGCCTCGTAATGCGGTTGTAGGTG"
w = "GCTTCTCCTAGGAGCTCCTCCAGTTTGAACTACGAATATTCGCATTACCTGGAACACTCAAGCCGGACCTGCGGAAGAGCTCTGGATTAGCGTCGACACTGCCTAGGTACCTGCATACCTTAAATTACGATCCTGGCGGCTTGATCCTGCTGATTCTGCTGAGTACAGTCAGTATCGTGCGTCCACTTCGACCCTATCTGGTCGTCGAATCGTATAATTCGCCACGTCCTCCCGTGTGACGTATACAGTTCACGGGTTCAAAATAGTAATCGTTACCCCGTTTCAGTCTGCGAACTTGGATTTACATTTGATCCTCTTGCGGGAAGTGATACCTGGAGAAGCGCCAGATGATTGTCCACCGTTACTTCGATCGGACAGGGAGCAATCTGCTTTCCTGTAAGCAGTGAAGACCTGAGTTAACTACTAGTGCTAGTGTCATGCCGCCAAAGTTGTTTCTTGCCGGAGTACTTGTGTGAGCGCTAACCCCTCTTATCGAGTGGTGATTTGCGATCCCGTTAGAAACCAAACTCTAAATATCAAGAGGCGCATGCGCGGCTTCCCGGACGGCCGGATGACAGACACGATGGTGAGCCATTAAACTATGTAAGTCGTCGTCAGCTATCAGTTCCAGTCGATATTATCGCGCAGAGTCGCACCTGCTACGGTAGCATTGCGCCAGTACAACGCTTACGCCCCTTCGCGAATCTATTTCGAAGCAATCGTTCAAACAATACTGCTTACTAAAGTGAAGCCCCCTTGAAAGGGACTACTGAGGCACTGCGTGCCCACTCTGACAGAATGGGAGAGAAACAGATCACTCAGTGATCGCGATCACCTGTTTAACGAGTAGAATGCACCAAGGGATTGAGTAGGTATTTTATCGGCCATCGAATAAGATGTAAACCAATCCACCTTGGGAAGCCATCTATACCTTACGATCTTTCTTGTGTTCGTGAGCAGATCGACAAGTATTCGACCAGAGGCATAGATGGAGCTGGAAAACTCACGGACTAATCCAGTGGACAGGATTCGCATATCTTGGTCATCGTGTCTGCCTGGACGCAAGTGAAGAAAAGTCGGGGGCGTCCGAATGTATTGTTGACTTGAAAACAATTGCGAATATAAGGAAATCTCTTGGGAGTTGTATTGGTCAGTGTATGTAGAATTCCACATCATAGCCTCGTCAGTAGTCCTGCGGTAAGTCCTCCGCCCAGACCTTAAAGAAATTAGTGGAGCCTGGCGGCGCTTGCAGCCACCACTGTTATCCAGCGAAATACCTCCGCGCTAGAATGTAATTCAGCCGCTGATAAAGGAGATTTCAAGAGACCAATCCGTAGGCCTTTAATAACTAGTGAACATATAGTACTGAAACAACAGCCGAGCATTTACTAGAGTTGAGAAGACTTCTCCTAGAGCACGGTCACAGATTCGCCGGGATAAGCTCGCCGCCTATTAGAAGTACTTGGACGTAAGTAAAGAACTGTATTGGCCTCTGGCACTGAGGTTTAGTAGCTGAGGGTTGACGTGTGTGTAGGACAGCCTATCGTTACTAACCAGGGGATAGCTCATAACTTGGTGGCGTTCCGTTGAGGATCAGGTTCCCGCGTTTCCGGACTCCACGTAGTAATTGAAAAGTTTCGACCGGGACTACGGAAAGTACGCGTGGGCCCAATTACGGACCGGGTATACGTCCGACGGGCCGGCAGTGTTACGACTGTCTAAGCGGTGTGGGTTTCCCAACTCGATGGTCCCTTTTTGAACGGGCGACGGTTGCCGTGGCCACGGCCTAATAGGGTCGGCCTTCTCGAATATAATAGGTG"

score, av, aw = hirschberg_alignment(v, w, match, mismatch, indel)

print("Score:", score)
print(av)
print(aw)

Score: -166
GCTTCTATCAACGAATCTTCGCATTACATGTA-AAACTCAAGCCGGAACCAGATCTCTGGAT-TAG-CGTCGACACGATTTATGCCCAGGTAACTGCATGCTATATTTAGGCGGGGATCCGTAATT-CTCGGAAGACA-AGAGGCCCTGCTGATTCTGCTATGTCATGTCAGGGTATCGTGCGTCCATTTCATAGGCAATTGTGGTCTAAGATACATCGTGAAATGTCCTCGACGTCCTCCCGTGTGACGTATACAGTTCACGATTTTGCTGGTTCAAGTAGTAATCGTGGCAAAGACGTTGGCGCTTGGATTTTATCGGCACTATTGATCCTCTTGCGGGAAGT-GATACCAGAGTGGCGCCAGATGATTGTTA-TTCGATCGGATAGGGAGCAATCTGCTTTCCTG-A--C---C--GACCTGAGTTAACTACTA-TG-T--TGTCA-G--GCCAAAGTTGTTTATTGCCTCAGTACTTGTGTGA-TATAGTCTGATTTGCGATCGTTACGAATTGC-ATCGACTT-T--TCGAAAAT-T-GTCACCGA-AGGCGCACGCGCGG-ATA-C-TA--ACCTTCCCACGGGCT-GATG-ACAGAGGAT-GAC-AT-TACAAC-T-TTC-G-TATCAGTTCCAG-AATTCCTATCGCGCACGTCTGCTCCT--TACGGTTGCAATGCGCCAGT--T--GCTTACGCCCCTTCGCGCATGGCAATCGT-TCAAACAATACTGCAAGAGCTCTTATTACAGCTTGAAACATACGAGTAA-GTACTGCTGAGGGGCTCCTTGCGTGCCTACTCTGCCGTAATGGGAGAAACAGATCCAATGCATCACTCCAAGTGAGCGCGGCTCACCTGTTTAACG-AGTAGGAAAACTTGCACCAGGGATTGCGTAGGT-ATTTTATCGGCTATCGAATCTCTATAACCAATCCAGCTGCATCCTACGAT-TTTC-T-TG-T-GTG-GCAGATCGCCAAGT

In [2]:
def lcs3(a, b, c):
    """
    Computes:
      1) length of the LCS of strings a, b, c
      2) a multiple alignment consistent with it
    """

    n1, n2, n3 = len(a), len(b), len(c)

    # DP table: dp[i][j][k] = LCS length of a[:i], b[:j], c[:k]
    dp = [[[0] * (n3 + 1) for _ in range(n2 + 1)] for __ in range(n1 + 1)]

    # Fill DP
    for i in range(1, n1 + 1):
        for j in range(1, n2 + 1):
            for k in range(1, n3 + 1):
                if a[i-1] == b[j-1] == c[k-1]:
                    dp[i][j][k] = dp[i-1][j-1][k-1] + 1
                else:
                    dp[i][j][k] = max(
                        dp[i-1][j][k],
                        dp[i][j-1][k],
                        dp[i][j][k-1],
                        dp[i-1][j-1][k],
                        dp[i-1][j][k-1],
                        dp[i][j-1][k-1]
                    )

    # ----------------------------------------
    # Backtrack to recover alignment
    # ----------------------------------------
    i, j, k = n1, n2, n3
    A, B, C = [], [], []

    while i > 0 or j > 0 or k > 0:
        # If all characters match, this is part of the LCS
        if i > 0 and j > 0 and k > 0 and a[i-1] == b[j-1] == c[k-1] \
           and dp[i][j][k] == dp[i-1][j-1][k-1] + 1:
            A.append(a[i-1])
            B.append(b[j-1])
            C.append(c[k-1])
            i -= 1; j -= 1; k -= 1
        else:
            # Otherwise move in the direction that preserves dp value
            moved = False

            if i > 0 and dp[i][j][k] == dp[i-1][j][k]:
                A.append(a[i-1]); B.append("-"); C.append("-")
                i -= 1; moved = True

            elif j > 0 and dp[i][j][k] == dp[i][j-1][k]:
                A.append("-"); B.append(b[j-1]); C.append("-")
                j -= 1; moved = True

            elif k > 0 and dp[i][j][k] == dp[i][j][k-1]:
                A.append("-"); B.append("-"); C.append(c[k-1])
                k -= 1; moved = True

            elif i > 0 and j > 0 and dp[i][j][k] == dp[i-1][j-1][k]:
                A.append(a[i-1]); B.append(b[j-1]); C.append("-")
                i -= 1; j -= 1; moved = True

            elif i > 0 and k > 0 and dp[i][j][k] == dp[i-1][j][k-1]:
                A.append(a[i-1]); B.append("-"); C.append(c[k-1])
                i -= 1; k -= 1; moved = True

            elif j > 0 and k > 0 and dp[i][j][k] == dp[i][j-1][k-1]:
                A.append("-"); B.append(b[j-1]); C.append(c[k-1])
                j -= 1; k -= 1; moved = True

            if not moved:  # Safety fallback
                break

    # Reverse to correct orientation
    return dp[n1][n2][n3], "".join(reversed(A)), "".join(reversed(B)), "".join(reversed(C))


# --------------------------
# Example
# --------------------------

a = "CGGAACTGGT"
b = "TGAGACGGTA"
c = "TGCGACGGCT"

L, A, B, C = lcs3(a, b, c)

print(L)
print(A)
print(B)
print(C)

7
--CG--GAACTGG-T-
-T-G-AG-AC-GG-TA
T--GC-G-AC-GGCT-
